# Script match_batch_waveform:

This script uses the template event we built in the previous script to detect new earthquakes using template matching. 
This script enables comparing 3-component waveforms and detect the number of matched event detected, further it support output of the detected event time of each day. 

In [1]:
import sys
import os
sys.path.append(os.getcwd())

import h5py as h5
import numpy as np
import utils

from obspy.core import UTCDateTime as udt
import matplotlib.pyplot as plt
import fast_matched_filter as fmf
from time import time as give_time
import logging
from datetime import datetime


# 获取当前时间并格式化为字符串
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f'matched_filter_{current_time}.log'

# 配置日志格式和级别
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename=log_filename,  # 使用带时间戳的日志文件名
                    filemode='w')
# 创建控制台输出
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter('%(levelname)s - %(message)s')
console.setFormatter(formatter)
logging.getLogger('').addHandler(console)

You might have received a warning message complaining about the cuda library not being here. This happens if you could not compile the C/cuda code when installing Fast Matched Filter (FMF). This is not a problem if you do not have any Nvidia GPUs, but you might want to recompile FMF if you want to beneficiate from GPUs. 

## Load the data and the template that we need to analyze

In [28]:
import h5py as h5
data = h5.File("./logs/matched_filter_results_threshold0.5_weight1.0_0.0_0.0_20250606_141252.h5","r")
print(data.keys())
for i in data.keys():
    print(data[i]['matched_count'][()])

<KeysViewHDF5 ['template_15021000.h5', 'template_15029300.h5', 'template_15059524.h5', 'template_15355949.h5', 'template_15390050.h5', 'template_15625774.h5', 'template_15643044.h5', 'template_15754899.h5', 'template_15992000.h5', 'template_16181250.h5', 'template_16206050.h5', 'template_16229300.h5', 'template_16232750.h5', 'template_16240899.h5']>
1
30
81
89
132
193
219
268
285
907
913
1213
1214
1214


In [2]:
h5_dir = './'
template_dir = './templates/'

sac_events = [f for f in os.listdir(h5_dir) if f.endswith('.h5')]
template_events = [f for f in os.listdir(template_dir) if f.endswith('.h5')]

#### parameter to run the code

In [3]:
threshold = 0.7
logging.info(f'Parameter to run matched filter search: threshold = {threshold}')

INFO - Parameter to run matched filter search: threshold = 0.7


In [4]:
matched_count = []
matched_event_list = []

for template_event in template_events:
	template = utils.load_template(template_event, path='./templates/')
	logging.info(f'successfully load template: {template_event}')
	# format the inputs for fmf by adding a new dimension
	# with 1 element (because this example only uses one template)
	template_array = template['waveforms'][np.newaxis, :]
	moveouts = np.hstack( (template['moveouts_S'].reshape(-1, 1),
						template['moveouts_S'].reshape(-1, 1),
						template['moveouts_P'].reshape(-1, 1)) )
	moveout_array = moveouts[np.newaxis, :]
	# fmf requires a weight matrix used to compute the weighted correlation
	# coefficient sum
	weight_array = np.ones_like(moveout_array, dtype=np.float32)
	n_stations = weight_array.shape[1]
	n_components = weight_array.shape[2]
	# normalize so that the max value is 1 (optional)
	weight_array /= np.float32(n_stations * n_components)
	# fmf needs two extra arguments:
	matched_filter_step = 1 # if set to 1, the sliding windows are taken every sample
	architecture = 'cpu' # run fmf on GPUs (other option is 'cpu')
	for sac_event in sac_events:
		data = utils.load_data(sac_event)
		logging.info(f'successfully load sacdata: {sac_event}')
		t_start = give_time()
		cc_sum = fmf.matched_filter(template_array,
									moveout_array,
									weight_array,
									data['waveforms'],
									matched_filter_step,
									arch=architecture)
		logging.info(f'successfully matched filter: {sac_event}')
		t_end = give_time()
		logging.info(f'Match template:{template_event} with sac data:{sac_event} , consuming {t_end-t_start}seconds')
		time = np.linspace(0., float(cc_sum.shape[1]) / data['metadata']['sampling_rate'], cc_sum.shape[1])
		matched_event_time = np.abs(cc_sum[0,:]) > threshold
		matched_event_num = sum(matched_event_time)
		matched_event_list.append(matched_event_num)
	matched_count.append(sum(matched_event_list))

INFO - successfully load template: template_15021000.h5
INFO - successfully load sacdata: 20230725_LX.Xs28.h5
INFO - successfully matched filter: 20230725_LX.Xs28.h5
INFO - Match template:template_15021000.h5 with sac data:20230725_LX.Xs28.h5 , consuming 0.8628284931182861seconds
INFO - successfully load sacdata: 20230726_LX.Xs28.h5
INFO - successfully matched filter: 20230726_LX.Xs28.h5
INFO - Match template:template_15021000.h5 with sac data:20230726_LX.Xs28.h5 , consuming 0.8644213676452637seconds
INFO - successfully load sacdata: 20230727_LX.Xs28.h5
INFO - successfully matched filter: 20230727_LX.Xs28.h5
INFO - Match template:template_15021000.h5 with sac data:20230727_LX.Xs28.h5 , consuming 0.9652559757232666seconds
INFO - successfully load sacdata: 20230728_LX.Xs28.h5
INFO - successfully matched filter: 20230728_LX.Xs28.h5
INFO - Match template:template_15021000.h5 with sac data:20230728_LX.Xs28.h5 , consuming 0.7605462074279785seconds
INFO - successfully load sacdata: 20230729_L

In [5]:
matched_count

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

## Plot the correlation coefficients

Note that the max value should be one, since we extracted the template event from this day and we are using **matched_filter_step** = 1. Note also that this is only possible when there are no rounding errors between the times used to extract the windows and the moveouts given to FMF (which is a common error!). Remember that we took care of rounding our travel times before extracting the windows, in order to have consistent window shifts and moveouts.

In [ ]:
# tune some plotting parameters
font = {'family': 'serif', 
        'size': 18}
plt.rc('font', **font)

In [ ]:
# plot the cc time series
figsize = (50, 12)
plt.figure('cc_sum', figsize=figsize)
time = np.linspace(0., float(cc_sum.shape[1]) / data['metadata']['sampling_rate'], cc_sum.shape[1])
#smart_plot = np.abs(cc_sum[0,:]) > 2.5 * np.std(cc_sum[0,:])
#print(smart_plot.shape)

#plt.plot(time[smart_plot], cc_sum[0, smart_plot], lw=0.5)
plt.plot(time, cc_sum[0], lw=0.5)
plt.axhline(1, lw=2, ls='--', color='k')
plt.xlabel('Time (s)')
plt.ylabel('Average correlation coefficient')
plt.title(template_name+'_ccsum:'+str(lowcut)+'-'+str(highcut)+'Hz')
#plt.xlim(time.min(), time.max())
#plt.ylim(-0.2,0.2)
plt.show()

If the plot looks odd, with missing values, this is because we discard most of the low values to make the plot more friendly for your computer.

In [ ]:
# plot the cc time series
figsize = (28, 12)
plt.figure('cc_sum', figsize=figsize)
time = np.linspace(0., float(cc_sum.shape[1]) / data['metadata']['sampling_rate'], cc_sum.shape[1])

template_start_index = template['origin_time']/250
plt.axvline(x=template_start_index, lw=2, ls='--', color='red',label=f'TemplateTime:{template_start_index}')
plt.legend()

plt.plot(time, cc_sum[0], lw=0.5)
plt.axhline(1, lw=2, ls='--', color='k')
plt.xlabel('Time (s)')
plt.ylabel('Average correlation coefficient')
plt.title(template_name+'_ccsum')
#plt.title(template_name+'_ccsum:'+str(lowcut)+'-'+str(highcut)+'Hz')
#plt.xlim(time.min(), time.max())
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 假设 time 和 smart_plot 已经定义
# 提取 smart_plot 为 True 的时间点
true_times = time[smart_plot]

# 创建直方图
plt.figure(figsize=(20, 8))
plt.hist(true_times, bins=30, edgecolor='black')
plt.title('Distribution of True Indices in Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
# 显示图形
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 假设 time 和 smart_plot 已经定义
# 提取 smart_plot 为 True 的时间点
true_times = time[smart_plot]

# 创建直方图
plt.figure(figsize=(20, 8))
plt.hist(true_times, bins=30, edgecolor='black')
plt.title('Distribution of True Indices in Time')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
# 显示图形
plt.show()